## 📝 Celda 1: Importaciones, Trazabilidad y Versiones del Entorno

In [5]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import sys
import fastapi
import joblib
import numpy as np
import pandas as pd
import pydantic
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

VERSIONES_LIBRERIAS = {
    "python": sys.version.split()[0],
    "scikit-learn": sklearn.__version__,
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "joblib": joblib.__version__,
    "pydantic": pydantic.__version__,
    "fastapi": fastapi.__version__,
}

print("=== VERSIONES DEL ENTORNO (.venv) ===")
for lib, ver in VERSIONES_LIBRERIAS.items():
    print(f"  {lib:<15}: {ver}")

=== VERSIONES DEL ENTORNO (.venv) ===
  python         : 3.14.3
  scikit-learn   : 1.9.1
  pandas         : 3.0.6
  numpy          : 2.5.3
  joblib         : 1.6.0
  pydantic       : 2.13.5
  fastapi        : 0.141.1


## Punto 1 — Carga de viviendas.csv y Selección de 12 Columnas de Entrada (1,460 filas)

> 📌 **Evidencia de cumplimiento del dataset:**
> Se cumple estrictamente con la regla de dimensiones requerida por el docente:
> * **Columnas de entrada:** 12 columnas utilizadas (mínimo requerido: entre 5 y 15).
> * **Filas de datos:** 1,460 filas registradas (mínimo requerido: 500).

In [6]:
RUTA_DATOS = Path("viviendas.csv")
assert RUTA_DATOS.exists(), f"No se encontró el archivo {RUTA_DATOS.resolve()}"

# Hash SHA-256 del dataset para trazabilidad (buena práctica de producción)
hash_dataset = hashlib.sha256(RUTA_DATOS.read_bytes()).hexdigest()
df_raw = pd.read_csv(RUTA_DATOS)

COLUMNAS_NUMERICAS = [
    "OverallQual",   # Calidad general del material y acabado (1 a 10)
    "GrLivArea",     # Área habitable sobre el nivel del suelo en pies²
    "GarageCars",    # Capacidad del garaje en cantidad de autos
    "TotalBsmtSF",   # Área total del sótano en pies²
    "FullBath",      # Baños completos sobre el nivel del suelo
    "YearBuilt",     # Año original de construcción
    "YearRemodAdd",  # Año de remodelación
    "LotArea",       # Superficie total del terreno en pies²
]

COLUMNAS_CATEGORICAS = [
    "KitchenQual",   # Calidad de la cocina (Ex, Gd, TA, Fa)
    "ExterQual",     # Calidad del material exterior (Ex, Gd, TA, Fa)
    "CentralAir",    # Aire acondicionado central (Y, N)
    "Neighborhood",  # Vecindario en Ames (25 categorías)
]

COLUMNAS_ENTRADA = COLUMNAS_NUMERICAS + COLUMNAS_CATEGORICAS
COLUMNA_OBJETIVO = "SalePrice"

df = df_raw[COLUMNAS_ENTRADA + [COLUMNA_OBJETIVO]].copy()

print(f"Archivo cargado               : {RUTA_DATOS.name} (SHA-256: {hash_dataset[:16]}...)")
print(f"Filas totales del dataset     : {df.shape[0]} (Requisito >= 500 filas)")
print(f"Columnas de entrada (X)       : {len(COLUMNAS_ENTRADA)} (Requisito entre 5 y 15 columnas)")
print(f"Valores nulos en las 13 cols  : {int(df.isnull().sum().sum())}")
df.head()

Archivo cargado               : viviendas.csv (SHA-256: 1e18addf81e5e4d3...)
Filas totales del dataset     : 1460 (Requisito >= 500 filas)
Columnas de entrada (X)       : 12 (Requisito entre 5 y 15 columnas)
Valores nulos en las 13 cols  : 0


,OverallQual,GrLivArea,GarageCars,TotalBsmtSF,FullBath,YearBuilt,YearRemodAdd,LotArea,KitchenQual,ExterQual,CentralAir,Neighborhood,SalePrice
0,7,1710,2,856,2,2003,2003,8450,Gd,Gd,Y,CollgCr,208500
1,6,1262,2,1262,2,1976,1976,9600,TA,TA,Y,Veenker,181500
2,7,1786,2,920,2,2001,2002,11250,Gd,Gd,Y,CollgCr,223500
3,7,1717,3,756,1,1915,1970,9550,Gd,TA,Y,Crawfor,140000
4,8,2198,3,1145,2,2000,2000,14260,Gd,Gd,Y,NoRidge,250000


## Punto 2 — Partición Train/Test, Entrenamiento del Pipeline y Evaluación con Datos No Vistos

### 📊 División del Dataset (80% / 20%)
Para garantizar una evaluación justa del modelo, los datos se dividieron de la siguiente manera:

* **Conjunto de Entrenamiento (80%):** Se asignaron **1,168 filas** para que el algoritmo aprenda los patrones.
* **Conjunto de Prueba/Oculto (20%):** Se reservaron **292 filas** completamente aisladas para evaluar el rendimiento del modelo con datos que nunca vio.


In [7]:
X = df[COLUMNAS_ENTRADA]
y = df[COLUMNA_OBJETIVO]

# Partición reproducible: 80% entrenamiento (1,168 filas) y 20% prueba no vista (292 filas)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

preprocesador = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            COLUMNAS_NUMERICAS,
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]),
            COLUMNAS_CATEGORICAS,
        ),
    ]
)

candidatos = {
    "Ridge Regression": Ridge(alpha=10.0),
    "Random Forest": RandomForestRegressor(
        n_estimators=300, max_depth=15, random_state=42, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=200, learning_rate=0.05, max_depth=4,
        min_samples_leaf=3, subsample=0.85, random_state=42
    ),
}

print(f"Evaluación en datos NO vistos (X_test = {len(X_test)} filas):")
for nombre, alg in candidatos.items():
    pipe_eval = Pipeline([("preprocesamiento", preprocesador), ("modelo", alg)])
    pipe_eval.fit(X_train, y_train)
    preds = pipe_eval.predict(X_test)
    cv_score = cross_val_score(pipe_eval, X_train, y_train, cv=5, scoring="r2").mean()
    print(
        f"  {nombre:<20} | CV R2: {cv_score:.4f} | Test R2: {r2_score(y_test, preds):.4f} "
        f"| RMSE: ${root_mean_squared_error(y_test, preds):,.2f} | MAE: ${mean_absolute_error(y_test, preds):,.2f}"
    )

Evaluación en datos NO vistos (X_test = 292 filas):
  Ridge Regression     | CV R2: 0.8057 | Test R2: 0.8491 | RMSE: $34,021.16 | MAE: $20,894.90
  Random Forest        | CV R2: 0.8249 | Test R2: 0.8967 | RMSE: $28,150.22 | MAE: $17,845.96
  Gradient Boosting    | CV R2: 0.8462 | Test R2: 0.9162 | RMSE: $25,348.27 | MAE: $16,587.64


## 🚀 Entrenamiento del Modelo Definitivo y Cálculo de Métricas Completas (Train vs. Test)

In [8]:
modelo_final = Pipeline([
    ("preprocesamiento", preprocesador),
    ("modelo", GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=4,
        min_samples_leaf=3,
        subsample=0.85,
        random_state=42,
    )),
])

modelo_final.fit(X_train, y_train)

pred_train = modelo_final.predict(X_train)
pred_test = modelo_final.predict(X_test)
cv_r2_train = cross_val_score(modelo_final, X_train, y_train, cv=5, scoring="r2").mean()

metricas_modelo = {
    "r2_test": round(float(r2_score(y_test, pred_test)), 4),
    "rmse_test": round(float(root_mean_squared_error(y_test, pred_test)), 2),
    "mae_test": round(float(mean_absolute_error(y_test, pred_test)), 2),
    "r2_train": round(float(r2_score(y_train, pred_train)), 4),
    "rmse_train": round(float(root_mean_squared_error(y_train, pred_train)), 2),
    "mae_train": round(float(mean_absolute_error(y_train, pred_train)), 2),
    "r2_cv_5fold": round(float(cv_r2_train), 4),
    "filas_train": int(X_train.shape[0]),
    "filas_test": int(X_test.shape[0]),
}

print("=== MÉTRICAS DEL MODELO FINAL ===")
print(json.dumps(metricas_modelo, indent=2))

=== MÉTRICAS DEL MODELO FINAL ===
{
  "r2_test": 0.9162,
  "rmse_test": 25348.27,
  "mae_test": 16587.64,
  "r2_train": 0.9628,
  "rmse_train": 14888.82,
  "mae_train": 10809.99,
  "r2_cv_5fold": 0.8462,
  "filas_train": 1168,
  "filas_test": 292
}


## 📦 Punto 3: Construcción del Bundle con `joblib`

Para garantizar la portabilidad del modelo y facilitar su despliegue, se procede a exportar tanto el modelo entrenado (**GradientBoostingRegressor**) como los artefactos de preprocesamiento necesarios en un único archivo serializado (*bundle*).

In [9]:
# 1. Diccionario de valores válidos indexado directamente por cada una de las 12 columnas
valores_validos = {}
for col in COLUMNAS_NUMERICAS:
    valores_validos[col] = {
        "tipo": "numerica",
        "min": float(df[col].min()),
        "max": float(df[col].max()),
        "mediana": float(df[col].median()),
    }

for col in COLUMNAS_CATEGORICAS:
    valores_validos[col] = {
        "tipo": "categorica",
        "valores_permitidos": sorted(df[col].dropna().unique().tolist()),
    }

# 2. Construcción del Bundle cumpliendo el 100% del Punto 3
fecha_utc = datetime.now(timezone.utc).isoformat()

bundle = {
    "modelo": modelo_final,
    "columnas": COLUMNAS_ENTRADA,
    "valores_validos": valores_validos,
    "metricas": metricas_modelo,
    "fecha_entrenamiento": fecha_utc,
    "versiones_librerias": VERSIONES_LIBRERIAS,
    # Metadatos adicionales de trazabilidad senior
    "target": COLUMNA_OBJETIVO,
    "tipo_problema": "regresion",
    "algoritmo": "GradientBoostingRegressor",
    "hash_datos_sha256": hash_dataset,
}

# 3. Guardado del bundle con joblib y respaldo de métricas en JSON
RUTA_BUNDLE = Path("modelo_bundle_viviendas.joblib")
joblib.dump(bundle, RUTA_BUNDLE)

RUTA_METRICAS = Path("metrics.json")
RUTA_METRICAS.write_text(
    json.dumps(
        {
            "fecha_entrenamiento": fecha_utc,
            "metricas": metricas_modelo,
            "columnas": COLUMNAS_ENTRADA,
            "versiones_librerias": VERSIONES_LIBRERIAS,
        },
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print(f"Bundle guardado en : {RUTA_BUNDLE.resolve()}")
print(f"Métricas JSON en   : {RUTA_METRICAS.resolve()}")

Bundle guardado en : C:\Users\Daniel\Desktop\Tarea_02_despliegue_de_modelos\modelo_bundle_viviendas.joblib
Métricas JSON en   : C:\Users\Daniel\Desktop\Tarea_02_despliegue_de_modelos\metrics.json


In [10]:
# 4. PUNTO 4: Verificación de carga y predicción 100% idéntica (Lote Test + Registro Individual)
bundle_verificado = joblib.load(RUTA_BUNDLE)

claves_exigidas = [
    "modelo",
    "columnas",
    "valores_validos",
    "metricas",
    "fecha_entrenamiento",
    "versiones_librerias",
]
for clave in claves_exigidas:
    assert clave in bundle_verificado, f"Falta la clave obligatoria: {clave}"

# Comprobación en todo el conjunto de prueba no visto (292 filas)
pred_original_lote = modelo_final.predict(X_test)
pred_recargado_lote = bundle_verificado["modelo"].predict(X_test)
assert np.allclose(pred_original_lote, pred_recargado_lote), "Diferencia detectada en lote."

# Comprobación con un registro individual (simulando petición JSON en FastAPI)
ejemplo_unitario = pd.DataFrame([X_test.iloc[0].to_dict()])
pred_unitaria_orig = float(modelo_final.predict(ejemplo_unitario)[0])
pred_unitaria_load = float(bundle_verificado["modelo"].predict(ejemplo_unitario)[0])
assert np.isclose(pred_unitaria_orig, pred_unitaria_load), "Diferencia en predicción unitaria."

print("\n=== VERIFICACIÓN DEL PUNTO 4 EXITOSA ===")
print("Claves presentes en el bundle :", list(bundle_verificado.keys()))
print("Predicción idéntica en 292 filas de Test :", np.allclose(pred_original_lote, pred_recargado_lote))
print(f"Predicción unitaria original  : ${pred_unitaria_orig:,.2f} USD")
print(f"Predicción unitaria recargada : ${pred_unitaria_load:,.2f} USD")


=== VERIFICACIÓN DEL PUNTO 4 EXITOSA ===
Claves presentes en el bundle : ['modelo', 'columnas', 'valores_validos', 'metricas', 'fecha_entrenamiento', 'versiones_librerias', 'target', 'tipo_problema', 'algoritmo', 'hash_datos_sha256']
Predicción idéntica en 292 filas de Test : True
Predicción unitaria original  : $139,223.61 USD
Predicción unitaria recargada : $139,223.61 USD
